# 02. Feature Engineering

This notebook converts the raw text (article, question, option) into a 20,013-dimensional feature matrix for machine learning.

## Feature Space
- **20,000 TF-IDF features:** Extracted from the combined text `article [SEP] question [SEP] option_text`.
- **13 Numeric/Similarity features:**
  1. Article-Question cosine similarity
  2. Article-Option cosine similarity
  3. Question-Option cosine similarity
  4. Article-Best-Sentence similarity
  5. Option-Best-Sentence similarity
  6. Question-Best-Sentence similarity
  7. Article length
  8. Question length
  9. Option length
  10. Question-Option word overlap
  11. Article-Option word overlap
  12. Option exact-match-in-article
  13. Option frequency-in-article

## Critical Constraint
- **Leakage Safety:** The TF-IDF vectorizer is fit **ONLY** on the training set. The validation and test sets are transformed using the fitted vectorizer.


In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, save_npz
import joblib
import nltk
from nltk.tokenize import sent_tokenize
import re

nltk.download('punkt_tab', quiet=True)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models_new")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Loading data...")
train_df = pd.read_csv(PROCESSED_DIR / "train_verification.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_verification.csv")
test_df = None
if (PROCESSED_DIR / "test_verification.csv").exists():
    test_df = pd.read_csv(PROCESSED_DIR / "test_verification.csv")


Loading data...


In [ ]:
import gc

def text_generator(df):
    """Yields the combined text row by row to prevent string concatenation memory spikes."""
    for _, row in df.iterrows():
        yield str(row['article']) + " [SEP] " + str(row['question']) + " [SEP] " + str(row.get('option_text', ''))

print("Fitting TF-IDF on Training set (Memory-Optimized)...")
# min_df=5 discards very rare n-grams early, preventing huge sparse matrices in memory during sorting.
# dtype=np.float32 saves 50% memory for the matrix values.
tfidf = TfidfVectorizer(
    max_features=20000, 
    ngram_range=(1, 2), 
    stop_words='english',
    min_df=5,
    dtype=np.float32
)

# Use fit_transform with the generator
X_train_tfidf = tfidf.fit_transform(text_generator(train_df))
gc.collect()

print(f"TF-IDF fitted. Vocab size: {len(tfidf.vocabulary_)}")
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.pkl")

print("Transforming Validation set...")
X_val_tfidf = tfidf.transform(text_generator(val_df))
gc.collect()

X_test_tfidf = None
if test_df is not None:
    print("Transforming Test set...")
    X_test_tfidf = tfidf.transform(text_generator(test_df))
    gc.collect()


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from nltk.tokenize import sent_tokenize
import time

def extract_numeric_features(df, tfidf_model):
    """Extracts the 13 numeric/similarity features using bulk vectorized operations (30x faster)."""
    start = time.time()
    
    articles = df['article'].fillna('').astype(str).tolist()
    questions = df['question'].fillna('').astype(str).tolist()
    options = df['option_text'].fillna('').astype(str).tolist()
    
    print("  [1/4] Bulk vectorizing strings (TF-IDF)...")
    A_mat = tfidf_model.transform(articles)
    Q_mat = tfidf_model.transform(questions)
    O_mat = tfidf_model.transform(options)
    
    print("  [2/4] Computing bulk element-wise cosine similarities...")
    # TF-IDF vectors are L2-normalized. Element-wise dot product of rows gives cosine similarity.
    sim_aq = A_mat.multiply(Q_mat).sum(axis=1).A1
    sim_ao = A_mat.multiply(O_mat).sum(axis=1).A1
    sim_qo = Q_mat.multiply(O_mat).sum(axis=1).A1
    
    print("  [3/4] Computing lengths and set overlaps...")
    len_a = np.array([len(set(a.split())) for a in articles])
    len_q = np.array([len(set(q.split())) for q in questions])
    len_o = np.array([max(1, len(set(o.split()))) for o in options])
    
    overlap_qo = np.array([len(set(q.split()) & set(o.split())) for q, o in zip(questions, options)]) / len_o
    overlap_ao = np.array([len(set(a.split()) & set(o.split())) for a, o in zip(articles, options)]) / len_o
    
    exact_match = np.array([1 if o in a else 0 for a, o in zip(articles, options)])
    freq = np.array([a.count(o) for a, o in zip(articles, options)])
    
    print("  [4/4] Caching unique articles for 'best sentence' feature...")
    unique_arts = df['article'].dropna().unique()
    art_to_smat = {}
    for a in unique_arts:
        sents = sent_tokenize(str(a))
        if sents:
            art_to_smat[a] = tfidf_model.transform(sents)
            
    print("        Computing best sentence logic...")
    features = []
    
    for i in range(len(df)):
        a_str = articles[i]
        s_mat = art_to_smat.get(a_str, None)
        
        sim_q_best = 0.0
        sim_a_best = 0.0
        sim_o_best = 0.0
        
        if s_mat is not None and s_mat.shape[0] > 0:
            q_row = Q_mat[i]
            # similarities between Q and all sentences in A
            sims = s_mat.multiply(q_row).sum(axis=1).A1
            best_idx = np.argmax(sims)
            
            best_s_vec = s_mat[best_idx]
            sim_q_best = sims[best_idx]
            # Since best_s_vec is a 1xV sparse matrix and A_mat[i] is 1xV, sum of multiply is dot product
            sim_a_best = best_s_vec.multiply(A_mat[i]).sum()
            sim_o_best = best_s_vec.multiply(O_mat[i]).sum()
            
        features.append([
            sim_aq[i], sim_ao[i], sim_qo[i],
            sim_a_best, sim_o_best, sim_q_best,
            len_a[i], len_q[i], len_o[i],
            overlap_qo[i], overlap_ao[i],
            exact_match[i], freq[i]
        ])
        
    end = time.time()
    print(f"  -> Extracted features for {len(df)} rows in {end-start:.1f} seconds.")
    return np.array(features)


In [ ]:
print("Extracting numeric features for Training set (this may take a while)...")
X_train_num = extract_numeric_features(train_df, tfidf)

print("Extracting numeric features for Validation set...")
X_val_num = extract_numeric_features(val_df, tfidf)

X_test_num = None
if test_df is not None:
    print("Extracting numeric features for Test set...")
    X_test_num = extract_numeric_features(test_df, tfidf)


In [ ]:
print("Combining features (TF-IDF + Numeric)...")
X_train_final = hstack([X_train_tfidf, X_train_num])
X_val_final = hstack([X_val_tfidf, X_val_num])

print(f"Final Train Shape: {X_train_final.shape}")
print(f"Final Val Shape: {X_val_final.shape}")

print("Saving feature matrices...")
save_npz(PROCESSED_DIR / "X_train_features.npz", X_train_final)
save_npz(PROCESSED_DIR / "X_val_features.npz", X_val_final)
np.save(PROCESSED_DIR / "y_train.npy", train_df['label'].values)
np.save(PROCESSED_DIR / "y_val.npy", val_df['label'].values)

# Save sample_ids for group-level evaluation later
np.save(PROCESSED_DIR / "train_sample_ids.npy", train_df['sample_id'].values)
np.save(PROCESSED_DIR / "val_sample_ids.npy", val_df['sample_id'].values)

if test_df is not None and X_test_tfidf is not None and X_test_num is not None:
    X_test_final = hstack([X_test_tfidf, X_test_num])
    print(f"Final Test Shape: {X_test_final.shape}")
    save_npz(PROCESSED_DIR / "X_test_features.npz", X_test_final)
    np.save(PROCESSED_DIR / "y_test.npy", test_df['label'].values)
    np.save(PROCESSED_DIR / "test_sample_ids.npy", test_df['sample_id'].values)

print("Feature engineering complete!")
